<a href="https://colab.research.google.com/github/palashDoshi1/CMPSC-448-Midterm-Project/blob/main/CMPSC_448_Midterm_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu130
True


In [2]:
import pandas as pd

URL = (
    "https://raw.githubusercontent.com/"
    "palashDoshi1/CMPSC-448-Midterm-Project/main/data/dataset.csv"
)

df = pd.read_csv(URL)
print(df.shape)
print(df.head())

(540, 5)
   prompt_id category   model  \
0          1   sports     gpt   
1          1   sports  claude   
2          1   sports  gemini   
3          2   sports     gpt   
4          2   sports  claude   

                                               input  \
0  How does quarterback performance compare with ...   
1  How does quarterback performance compare with ...   
2  How does quarterback performance compare with ...   
3  Explain how the NFL salary cap affects roster ...   
4  Explain how the NFL salary cap affects roster ...   

                                              output  
0  ## How does quarterback performance compare wi...  
1  Quarterback play matters more than coaching st...  
2  In modern football analytics, evaluating the r...  
3  ## How does the NFL salary cap affect roster c...  
4  The NFL has a hard cap. Teams can't exceed it,...  


In [3]:
required = ["prompt_id", "category", "model", "input", "output"]

assert set(required).issubset(df.columns)
assert not df[required].isna().any().any()

df["model"] = df["model"].str.strip().str.lower()

label_map = {"gpt": 0, "claude": 1, "gemini": 2}

assert set(df["model"]) == set(label_map)
assert not df.duplicated(["prompt_id", "model"]).any()

for pid, group in df.groupby("prompt_id"):
    assert len(group) == 3
    assert set(group["model"]) == set(label_map)
    assert group["input"].nunique() == 1
    assert group["category"].nunique() == 1

df["label"] = df["model"].map(label_map)

print("Responses:", len(df))
print("Prompts:", df["prompt_id"].nunique())
print(df["model"].value_counts())

Responses: 540
Prompts: 180
model
gpt       180
claude    180
gemini    180
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

SEED = 42

prompts = df[["prompt_id", "category"]].drop_duplicates()

train_prompts, remaining = train_test_split(
    prompts,
    test_size=0.30,
    stratify=prompts["category"],
    random_state=SEED
)

val_prompts, test_prompts = train_test_split(
    remaining,
    test_size=0.50,
    stratify=remaining["category"],
    random_state=SEED
)

train_df = df[df["prompt_id"].isin(train_prompts["prompt_id"])].copy()
val_df = df[df["prompt_id"].isin(val_prompts["prompt_id"])].copy()
test_df = df[df["prompt_id"].isin(test_prompts["prompt_id"])].copy()

assert set(train_df.prompt_id).isdisjoint(val_df.prompt_id)
assert set(train_df.prompt_id).isdisjoint(test_df.prompt_id)
assert set(val_df.prompt_id).isdisjoint(test_df.prompt_id)

print(len(train_df), len(val_df), len(test_df))

378 81 81
